# Phase 2: Reported-Event Exposure and Geography

This notebook is the analyst-facing interface for the Phase 2 pipeline. It consumes only the accepted Phase 1 v3 handoff and keeps point labels, duration sensitivity, pair-decision uncertainty, and exceptions separate.

The checked-in configuration is intentionally fail-closed until source coverage, a past-only cohort, a selected interval unit, and authoritative geographic provenance are supplied. An incomplete preflight is a controlled result and does not create exposure labels.

In [ ]:
import importlib
import json
from pathlib import Path
import sys

import pandas as pd

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'analysis').is_dir() and (path / 'data').is_dir()
)
analysis_dir = (repo_root / 'analysis').resolve()
analysis_path = str(analysis_dir)
sys.path = [entry for entry in sys.path if str(Path(entry or '.').resolve()) != analysis_path]
sys.path.insert(0, analysis_path)
sys.modules.pop('reported_event_exposure', None)
import reported_event_exposure
reported_event_exposure = importlib.reload(reported_event_exposure)

phase_1_dir = repo_root / 'analysis_outputs' / 'deduplication' / 'v3'
config_path = analysis_dir / 'reported_event_exposure_config.json'
output_dir = repo_root / 'analysis_outputs' / 'reported_event_exposure' / 'v1'

result = reported_event_exposure.run_phase_2(phase_1_dir, config_path, output_dir)
result.summary

## Phase 1 handoff

The pipeline validates the Phase 1 gate, ruleset, manifest row counts, schemas, pair outcomes, and one-to-one source disposition before any Phase 2 processing.

In [ ]:
gate_report = json.loads(result.artifact_paths['phase_2_gate_report'].read_text(encoding='utf-8'))
display(pd.Series(gate_report['phase1_handoff']))
display(pd.Series(result.validations))

## Coverage, geography, and cohort inputs

`no_report_observed` is permitted only inside documented coverage. Geographic membership must come from an approved, versioned source, and exposure is generated only after requested-region and past-only cohort filtering.

In [ ]:
config = json.loads(config_path.read_text(encoding='utf-8'))
display(pd.Series({
    'coverage_path': config.get('source_coverage_path'),
    'region_source_status': config.get('region_source', {}).get('approval_status'),
    'requested_region_ids': config.get('requested_region_ids'),
    'cohort_path': config.get('cohort_path'),
    'eligibility_as_of': config.get('eligibility_as_of'),
    'candidate_interval_units_hours': config.get('candidate_interval_units_hours'),
    'selected_interval_unit_hours': config.get('selected_interval_unit_hours'),
}))
gate_report.get('blockers', [])

## Phase 2 evidence and decision gate

When preflight is complete, these displays are populated from generated artifacts. Primary point labels use only `report_observed`, `no_report_observed`, and `unknown`; sensitivity fields never overwrite the primary label.

In [ ]:
display(gate_report)
if gate_report['status'] == 'complete':
    display(pd.read_csv(result.artifact_paths['interval_unit_comparison']))
    display(pd.read_csv(result.artifact_paths['uncertainty_diagnostics']))
    display(pd.read_csv(result.artifact_paths['region_assignment_diagnostics']))
    display(json.loads(result.artifact_paths['exposure_strategy_comparison'].read_text(encoding='utf-8')))